<a href="https://colab.research.google.com/github/rafliatha/scraping-tweetharvest/blob/main/Scraping_Twitter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================
# Twitter / X Authentication
# ============================

# Variabel ini digunakan untuk menyimpan Twitter (X) Auth Token.
# Auth Token berfungsi sebagai kunci autentikasi agar program
# dapat mengakses akun Twitter/X tertentu secara otomatis.

# Token ini biasanya didapatkan dengan cara:
# 1. Login ke akun Twitter/X
# 2. Mengambil auth token dari cookie browser
# 3. Menyalinnya (copy-paste) ke dalam variabel ini

# Catatan penting:
# - Auth Token bersifat rahasia (private)
# - Jangan dibagikan ke orang lain
# - Jangan diunggah ke repository publik (GitHub, dsb)

twitter_auth_token = '7656481e92f942d846a3ad6898fd002fcac084aa'  # Ganti dengan auth token akun masing-masing

In [ ]:
# ============================
# Install Required Packages
# ============================

# Menginstall library pandas
# Pandas digunakan untuk mengolah, membaca, dan menyimpan data
# (misalnya data tweet ke dalam bentuk tabel seperti CSV atau DataFrame)
!pip install pandas

# ============================
# Install Node.js
# ============================

# Tweet-harvest dibangun menggunakan Node.js,
# sehingga Node.js wajib diinstall agar tweet-harvest bisa dijalankan

# Update daftar package pada sistem
!sudo apt-get update

# Menginstall package pendukung untuk instalasi Node.js
# - ca-certificates : sertifikat keamanan
# - curl            : mengambil data dari internet
# - gnupg           : verifikasi keamanan (GPG key)
!sudo apt-get install -y ca-certificates curl gnupg

# Membuat direktori untuk menyimpan key repository Node.js
!sudo mkdir -p /etc/apt/keyrings

# Mengunduh dan menyimpan GPG key resmi Node.js
# Key ini digunakan untuk memastikan package Node.js yang diinstall aman dan valid
!curl -fsSL https://deb.nodesource.com/gpgkey/nodesource-repo.gpg.key | sudo gpg --dearmor -o /etc/apt/keyrings/nodesource.gpg

# Menambahkan repository Node.js versi 20 ke sistem
# NODE_MAJOR=20 berarti menggunakan Node.js versi 20 (versi stabil)
!NODE_MAJOR=20 && echo "deb [signed-by=/etc/apt/keyrings/nodesource.gpg] https://deb.nodesource.com/node_$NODE_MAJOR.x nodistro main" | sudo tee /etc/apt/sources.list.d/nodesource.list

# Update ulang package list setelah menambahkan repository baru
!sudo apt-get update

# Menginstall Node.js dari repository yang sudah ditambahkan
!sudo apt-get install nodejs -y

# ============================
# Check Installation
# ============================

# Mengecek versi Node.js untuk memastikan instalasi berhasil
!node -v
!npx --yes playwright install-deps

In [ ]:
# ============================
# HARVEST TWITTER DATA
# ============================

# Menentukan nama file output
# Data hasil crawling tweet akan disimpan dalam bentuk file CSV
filename = 'dataset.csv'

# Menentukan kata kunci pencarian
# - lang:id (Hanya bahasa Indonesia)
# - -filter:retweets (Membuang data duplikasi otomatis/retweet)
# - since:date (Membatasi rentang waktu awal)
# - until:date (Membatasi rentang waktu akhir)
search_keyword = 'depresi lang:id -filter:retweets -filter:links'

# Menentukan jumlah maksimal tweet yang akan diambil
# Disini, sistem akan mengambil hingga 1500 tweet terbaru
limit = 1500

# Menjalankan tweet-harvest menggunakan npx
# -o      : nama file output
# -s      : search keyword
# --tab   : jenis tab pencarian (LATEST = tweet terbaru)
# -l      : limit jumlah tweet
# --token : auth token Twitter/X untuk autentikasi akun
!npx -y tweet-harvest@latest -o "{filename}" -s "{search_keyword}" --tab "LATEST" -l {limit} --token {twitter_auth_token}

In [ ]:
# ==========================================
# LOAD DATAFRAME
# ==========================================
import pandas as pd

# Mengambil nama file dari Cell 1 sebelumnya
file_path = f"tweets-data/{filename}"

# Load data mentah ke dalam DataFrame
df = pd.read_csv(file_path, delimiter=",")

# Menghitung dan menampilkan jumlah tweet mentah
num_tweets = len(df)
print(f"Jumlah tweet mentah dalam dataframe adalah {num_tweets}.")

# Tampilkan preview data (opsional, untuk memastikan data terbaca)
display(df)

In [ ]:
# ==========================================
# PRAPEMROSESAN TEKS
# ==========================================
# Menginstall library untuk cek bahasa
!pip install langdetect

import re
from langdetect import detect, LangDetectException

# Menghapus data duplikat berdasarkan kolom teks asli (full_text)
df_clean = df.drop_duplicates(subset=['full_text']).copy()

# Fungsi Prapemrosesan Dasar
def bersihkan_teks(text):
    text = str(text)
    # Hapus Tautan (URL)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Hapus Sebutan Akun (@username)
    text = re.sub(r'\@\w+', '', text)
    # Hapus Tagar (#)
    text = re.sub(r'\#\w+', '', text)
    # Hapus karakter selain alfabet (angka, tanda baca, emoticon)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # Rapikan spasi ganda
    text = re.sub(r'\s+', ' ', text).strip()
    # Case Folding (Ubah ke huruf kecil)
    return text.lower()

# Menerapkan fungsi ke dalam kolom baru 'clean_text'
df_clean['clean_text'] = df_clean['full_text'].apply(bersihkan_teks)

# Membuang baris yang teksnya kosong setelah dibersihkan
df_clean = df_clean[df_clean['clean_text'] != '']
df_clean = df_clean.dropna(subset=['clean_text'])

# Mengecek bahaasa Indonesia
def apakah_bahasa_indonesia(text):
    try:
        return detect(text) == 'id'
    except LangDetectException:
        return False
print("Memulai proses penyaringan bahasa asing... (Ini mungkin memakan waktu beberapa detik)")

# Terapkan deteksi bahasa pada teks yang sudah dibersihkan
df_clean['is_indonesia'] = df_clean['clean_text'].apply(apakah_bahasa_indonesia)

# Buang (drop) semua baris yang terdeteksi sebagai bahasa asing
df_clean = df_clean[df_clean['is_indonesia'] == True]

# VALIDASI KONTEKS
# Kumpulan kata ganti orang pertama (Subjective)
kata_ganti = ['aku', 'saya', 'gue', 'gw', 'gua', 'ku', 'diriku']

def validasi_konteks(text):
    words = str(text).split()

    # Syarat A: HARUS mengandung minimal 1 kata ganti orang pertama
    is_first_person = any(kata in words for kata in kata_ganti)

    # Lolos jika memenuhi kedua syarat
    return is_first_person

print("[-] Memvalidasi sudut pandang orang pertama dan mengeksklusi teks informatif/iklan...")
df_clean['lolos_konteks'] = df_clean['clean_text'].apply(validasi_konteks)

# Membuang baris yang gagal validasi konteks
df_clean = df_clean[df_clean['lolos_konteks'] == True]

# Menghitung jumlah data bersih
jumlah_data_bersih = len(df_clean)

print("="*50)
print("TAHAP PRAPEMROSESAN SELESAI")
print("="*50)
print(f"Jumlah Data Bersih : {jumlah_data_bersih} cuitan (Catat angka ini ke Spreadsheet!)")
print("\nPreview Teks Asli vs Teks Bersih:")
display(df_clean[['full_text', 'clean_text']].head(5))

In [ ]:
# ==========================================
# PELABELAN & EKSPOR DATA
# ==========================================
from google.colab import files

# Ketik 1 (Jika mencari kata stres/depresi dll)
# Ketik 0 (Jika mencari kata bahagia/antusias dll)
label_biner = 0

# Kolom 'label' yang murni berisi angka 1 atau 0 untuk pelatihan IndoBERT
df_clean['kategori_leksikon'] = 'netral'
df_clean['label'] = label_biner

# Memilih kolom final
df_final = df_clean[['full_text', 'clean_text', 'kategori_leksikon', 'label']].copy()

# Ekspor ke CSV dan Unduh
nama_file_bersih = 'dataset_berlabel.csv' # Edit sesuai kata kunci
df_final.to_csv(nama_file_bersih, index=False, sep=';')

print("="*50)
print("PELABELAN DAN EKSPOR SELESAI")
print("="*50)
print(f"File Output : {nama_file_bersih}\n")
display(df_final.head(3))

# Mengunduh otomatis
files.download(nama_file_bersih)